In [0]:
%sql
-- Set up widget for mode parameter
CREATE WIDGET TEXT mode DEFAULT 'full_refresh';

-- Set up schema
CREATE SCHEMA IF NOT EXISTS workspace.promotion_sql;
USE workspace.promotion_sql;

SELECT 'Pipeline initialized' AS status, '${mode}' AS mode;

In [0]:
%sql
-- Create watermark and log tables
CREATE TABLE IF NOT EXISTS workspace.promotion_sql.pipeline_watermarks (
    table_name      STRING,
    last_year       INT,
    last_week       INT,
    rows_processed  BIGINT,
    run_status      STRING,
    run_mode        STRING,
    updated_at      TIMESTAMP
)
USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.promotion_sql.pipeline_run_log (
    run_id       STRING,
    run_mode     STRING,
    stage        STRING,
    rows_out     BIGINT,
    status       STRING,
    error_msg    STRING,
    started_at   TIMESTAMP,
    finished_at  TIMESTAMP
)
USING DELTA;

SELECT 'Watermark tables ready' AS status;

In [0]:
%sql
-- Load Product dimension
CREATE OR REPLACE TABLE workspace.promotion_sql.bronze_product
USING DELTA AS
SELECT 
  *,
  CURRENT_TIMESTAMP() AS _ingest_time,
  _metadata.file_path AS _source_file,
  CURRENT_DATE() AS _batch_date
FROM read_files(
  '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Product.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

-- Load Store dimension
CREATE OR REPLACE TABLE workspace.promotion_sql.bronze_store
USING DELTA AS
SELECT 
  *,
  CURRENT_TIMESTAMP() AS _ingest_time,
  _metadata.file_path AS _source_file,
  CURRENT_DATE() AS _batch_date
FROM read_files(
  '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Store.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

-- Load Date dimension
CREATE OR REPLACE TABLE workspace.promotion_sql.bronze_date
USING DELTA AS
SELECT 
  *,
  CURRENT_TIMESTAMP() AS _ingest_time,
  _metadata.file_path AS _source_file,
  CURRENT_DATE() AS _batch_date
FROM read_files(
  '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Date.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

-- Load Promotion dimension
CREATE OR REPLACE TABLE workspace.promotion_sql.bronze_promotion
USING DELTA AS
SELECT 
  *,
  CURRENT_TIMESTAMP() AS _ingest_time,
  _metadata.file_path AS _source_file,
  CURRENT_DATE() AS _batch_date
FROM read_files(
  '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Promotion.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

SELECT 'Dimensions loaded' AS status,
  (SELECT COUNT(*) FROM bronze_product) AS products,
  (SELECT COUNT(*) FROM bronze_store) AS stores,
  (SELECT COUNT(*) FROM bronze_date) AS dates,
  (SELECT COUNT(*) FROM bronze_promotion) AS promotions;

In [0]:
%sql
-- Get watermark for incremental mode
CREATE OR REPLACE TEMPORARY VIEW watermark AS
SELECT 
  COALESCE(MAX(last_year), 0) AS wm_year,
  COALESCE(MAX(CASE WHEN last_year = (SELECT MAX(last_year) FROM workspace.promotion_sql.pipeline_watermarks WHERE table_name = 'bronze_sales') 
                    THEN last_week ELSE 0 END), 0) AS wm_week
FROM workspace.promotion_sql.pipeline_watermarks
WHERE table_name = 'bronze_sales' AND run_status = 'SUCCESS';

-- Create table if not exists
CREATE TABLE IF NOT EXISTS workspace.promotion_sql.bronze_sales (
  Year INT,
  WeekNumber INT,
  StoreID STRING,
  Product STRING,
  Price DOUBLE,
  OnFlyer STRING,
  Discount STRING,
  Units INT,
  SalesAmt DOUBLE,
  GrossMargin DOUBLE,
  Transactions INT,
  _rescued_data STRING,
  _ingest_time TIMESTAMP,
  _source_file STRING,
  _batch_date DATE
)
USING DELTA;

-- For full refresh: delete all existing data
DELETE FROM workspace.promotion_sql.bronze_sales WHERE LOWER(TRIM('${mode}')) = 'full_refresh';

-- Prepare new data based on mode
CREATE OR REPLACE TEMPORARY VIEW new_sales_data AS
SELECT 
  CAST(raw.Year AS INT) AS Year,
  CAST(raw.WeekNumber AS INT) AS WeekNumber,
  raw.StoreID,
  raw.Product,
  CAST(raw.Price AS DOUBLE) AS Price,
  raw.OnFlyer,
  raw.Discount,
  CAST(raw.Units AS INT) AS Units,
  CAST(raw.SalesAmt AS DOUBLE) AS SalesAmt,
  CAST(raw.GrossMargin AS DOUBLE) AS GrossMargin,
  CAST(raw.Transactions AS INT) AS Transactions,
  raw._rescued_data,
  CURRENT_TIMESTAMP() AS _ingest_time,
  raw._metadata.file_path AS _source_file,
  CURRENT_DATE() AS _batch_date
FROM read_files(
  '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Sales.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
) raw
CROSS JOIN watermark wm
WHERE 
  -- Full refresh: load all data
  (LOWER(TRIM('${mode}')) = 'full_refresh')
  OR
  -- Incremental: load only new data after watermark
  (LOWER(TRIM('${mode}')) = 'incremental'
   AND (CAST(raw.Year AS INT) > wm.wm_year 
        OR (CAST(raw.Year AS INT) = wm.wm_year AND CAST(raw.WeekNumber AS INT) > wm.wm_week)));

-- Insert new data (works for both modes)
INSERT INTO workspace.promotion_sql.bronze_sales
SELECT * FROM new_sales_data;

-- Update watermark
INSERT INTO workspace.promotion_sql.pipeline_watermarks
SELECT 
  'bronze_sales' AS table_name,
  MAX(Year) AS last_year,
  MAX(CASE WHEN Year = (SELECT MAX(Year) FROM workspace.promotion_sql.bronze_sales) 
           THEN WeekNumber ELSE 0 END) AS last_week,
  COUNT(*) AS rows_processed,
  'SUCCESS' AS run_status,
  LOWER(TRIM('${mode}')) AS run_mode,
  CURRENT_TIMESTAMP() AS updated_at
FROM workspace.promotion_sql.bronze_sales;

-- Log the run
INSERT INTO workspace.promotion_sql.pipeline_run_log
SELECT
  DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMddHHmmss') AS run_id,
  LOWER(TRIM('${mode}')) AS run_mode,
  'bronze_sales' AS stage,
  COUNT(*) AS rows_out,
  'SUCCESS' AS status,
  '' AS error_msg,
  CURRENT_TIMESTAMP() AS started_at,
  CURRENT_TIMESTAMP() AS finished_at
FROM workspace.promotion_sql.bronze_sales;

SELECT 
  'Sales fact loaded' AS status, 
  LOWER(TRIM('${mode}')) AS mode,
  COUNT(*) AS total_rows,
  (SELECT wm_year FROM watermark) AS watermark_year,
  (SELECT wm_week FROM watermark) AS watermark_week
FROM bronze_sales;

In [0]:
%sql
-- Create Silver Product Dimension
CREATE OR REPLACE TABLE workspace.promotion_sql.silver_dim_product
USING DELTA AS
WITH cleaned_product AS (
    SELECT
        INITCAP(TRIM(Product)) AS Product,
        INITCAP(TRIM(Brand)) AS Brand,
        INITCAP(TRIM(Category)) AS Category,
        TRIM(Size) AS Size,
        INITCAP(TRIM(Supplier)) AS Supplier,
        CAST(UnitCost AS DOUBLE) AS UnitCost,
        ROW_NUMBER() OVER (PARTITION BY TRIM(Product) ORDER BY TRIM(Product)) AS rn
    FROM workspace.promotion_sql.bronze_product
)
SELECT
    ROW_NUMBER() OVER (ORDER BY Product) AS ProductKey,
    Product, Brand, Category, Size, Supplier, UnitCost
FROM cleaned_product
WHERE rn = 1;

SELECT 'Product dimension created' AS status, COUNT(*) AS rows FROM silver_dim_product;

In [0]:
%sql
-- Create Silver Store Dimension
CREATE OR REPLACE TABLE workspace.promotion_sql.silver_dim_store
USING DELTA AS
WITH cleaned_store AS (
    SELECT
        TRIM(StoreID) AS StoreID,
        INITCAP(TRIM(StoreName)) AS StoreName,
        INITCAP(TRIM(City)) AS City,
        INITCAP(TRIM(Province)) AS Province,
        UPPER(TRIM(ProvinceAbbrev)) AS ProvinceAbbrev,
        INITCAP(TRIM(Country)) AS Country,
        ROW_NUMBER() OVER (PARTITION BY TRIM(StoreID) ORDER BY TRIM(StoreID)) AS rn
    FROM workspace.promotion_sql.bronze_store
)
SELECT
    ROW_NUMBER() OVER (ORDER BY StoreID) AS StoreKey,
    StoreID, StoreName, City, Province, ProvinceAbbrev, Country
FROM cleaned_store
WHERE rn = 1;

SELECT 'Store dimension created' AS status, COUNT(*) AS rows FROM silver_dim_store;

In [0]:
%sql
-- Create Silver Date Dimension
CREATE OR REPLACE TABLE workspace.promotion_sql.silver_dim_date
USING DELTA AS
WITH cleaned_date AS (
    SELECT
        CAST(Year AS INT) AS Year,
        CAST(WeekNumber AS INT) AS WeekNumber,
        CAST(MonthNumber AS INT) AS MonthNumber,
        CONCAT('FY', REGEXP_EXTRACT(FiscalYear, '(\\d{4})', 1)) AS FiscalYear,
        INITCAP(TRIM(Month)) AS Month,
        WeekStartDate,
        CASE
            WHEN Quarter IS NOT NULL THEN Quarter
            WHEN CAST(MonthNumber AS INT) IN (2, 3, 4) THEN 'Q1'
            WHEN CAST(MonthNumber AS INT) IN (5, 6, 7) THEN 'Q2'
            WHEN CAST(MonthNumber AS INT) IN (8, 9, 10) THEN 'Q3'
            ELSE 'Q4'
        END AS Quarter,
        ROW_NUMBER() OVER (PARTITION BY CAST(Year AS INT), CAST(WeekNumber AS INT) 
                          ORDER BY CAST(Year AS INT), CAST(WeekNumber AS INT)) AS rn
    FROM workspace.promotion_sql.bronze_date
)
SELECT
    CAST(Year * 100 + WeekNumber AS INT) AS DateKey,
    Year, WeekNumber, WeekStartDate, Month, MonthNumber, Quarter, FiscalYear
FROM cleaned_date
WHERE rn = 1;

SELECT 'Date dimension created' AS status, COUNT(*) AS rows FROM silver_dim_date;

In [0]:
%sql
-- Create Silver Promotion Dimension
CREATE OR REPLACE TABLE workspace.promotion_sql.silver_dim_promotion
USING DELTA AS
WITH cleaned_promotion AS (
    SELECT
        TRIM(PromotionName) AS PromotionName,
        INITCAP(TRIM(OnFlyer)) AS OnFlyer,
        CASE
            WHEN TRIM(Discount) LIKE '%\%' THEN
                TRY_CAST(REGEXP_EXTRACT(TRIM(Discount), '([\\d\\.]+)', 1) AS DOUBLE) / 100
            WHEN TRY_CAST(Discount AS DOUBLE) > 1 THEN
                TRY_CAST(Discount AS DOUBLE) / 100
            ELSE TRY_CAST(Discount AS DOUBLE)
        END AS Discount,
        INITCAP(TRIM(PromotionType)) AS PromotionType,
        CASE
            WHEN TRIM(DiscountTier) IS NOT NULL AND TRIM(DiscountTier) != '' THEN INITCAP(TRIM(DiscountTier))
            WHEN TRY_CAST(Discount AS DOUBLE) >= 0.30 THEN 'Deep'
            WHEN TRY_CAST(Discount AS DOUBLE) > 0.00 THEN 'Mid'
            ELSE 'None'
        END AS DiscountTier,
        ROW_NUMBER() OVER (PARTITION BY TRIM(PromotionName) ORDER BY TRIM(PromotionName)) AS rn
    FROM workspace.promotion_sql.bronze_promotion
)
SELECT
    ROW_NUMBER() OVER (ORDER BY PromotionName) AS PromotionKey,
    PromotionName, OnFlyer, Discount, PromotionType, DiscountTier
FROM cleaned_promotion
WHERE rn = 1;

SELECT 'Promotion dimension created' AS status, COUNT(*) AS rows FROM silver_dim_promotion;

In [0]:
%sql
-- Create or update Silver Sales Fact based on mode
CREATE TABLE IF NOT EXISTS workspace.promotion_sql.silver_fact_sales (
    ProductKey INT,
    StoreKey INT,
    DateKey INT,
    PromotionKey INT,
    Year INT,
    WeekNumber INT,
    ActualPrice DOUBLE,
    DiscountPct DOUBLE,
    UnitsSold INT,
    SalesAmt DOUBLE,
    GrossMargin DOUBLE,
    Transactions INT,
    IsBelowCost INT
)
USING DELTA;

-- Prepare cleaned sales data
CREATE OR REPLACE TEMPORARY VIEW cleaned_sales_data AS
WITH cleaned_sales AS (
    SELECT
        INITCAP(TRIM(Product)) AS Product,
        INITCAP(TRIM(OnFlyer)) AS OnFlyer,
        CASE
            WHEN TRIM(Discount) LIKE '%\%' THEN
                TRY_CAST(REGEXP_EXTRACT(TRIM(Discount), '([\\d\\.]+)', 1) AS DOUBLE) / 100
            WHEN TRY_CAST(Discount AS DOUBLE) > 1 THEN
                TRY_CAST(Discount AS DOUBLE) / 100
            ELSE TRY_CAST(Discount AS DOUBLE)
        END AS Discount,
        Year,
        WeekNumber,
        TRIM(StoreID) AS StoreID,
        Price,
        Units,
        SalesAmt,
        GrossMargin,
        Transactions,
        ROW_NUMBER() OVER (PARTITION BY Year, WeekNumber, TRIM(StoreID), INITCAP(TRIM(Product)) 
                           ORDER BY Year, WeekNumber) AS rn
    FROM workspace.promotion_sql.bronze_sales
),
with_promotion_name AS (
    SELECT
        *,
        CASE
            WHEN Discount = 0 THEN 'No Promotion'
            WHEN OnFlyer = 'Yes' THEN CONCAT(CAST(Discount * 100 AS INT), '% Off + Flyer')
            ELSE CONCAT(CAST(Discount * 100 AS INT), '% Off')
        END AS PromotionName
    FROM cleaned_sales
    WHERE rn = 1
)
SELECT
    p.ProductKey,
    s.StoreKey,
    d.DateKey,
    pr.PromotionKey,
    f.Year,
    f.WeekNumber,
    COALESCE(f.Price, ROUND(f.SalesAmt / NULLIF(f.Units, 0), 2)) AS ActualPrice,
    f.Discount AS DiscountPct,
    f.Units AS UnitsSold,
    f.SalesAmt,
    f.GrossMargin,
    f.Transactions,
    CASE WHEN f.GrossMargin < 0 THEN 1 ELSE 0 END AS IsBelowCost
FROM with_promotion_name f
LEFT JOIN workspace.promotion_sql.silver_dim_product p ON f.Product = p.Product
LEFT JOIN workspace.promotion_sql.silver_dim_store s ON f.StoreID = s.StoreID
LEFT JOIN workspace.promotion_sql.silver_dim_date d ON f.Year = d.Year AND f.WeekNumber = d.WeekNumber
LEFT JOIN workspace.promotion_sql.silver_dim_promotion pr ON f.PromotionName = pr.PromotionName;

-- Prepare data for each mode
CREATE OR REPLACE TEMPORARY VIEW full_refresh_silver AS
SELECT * FROM cleaned_sales_data
WHERE LOWER(TRIM('${mode}')) = 'full_refresh';

CREATE OR REPLACE TEMPORARY VIEW incremental_silver AS
SELECT * FROM cleaned_sales_data
WHERE LOWER(TRIM('${mode}')) = 'incremental';

-- Full refresh: Replace entire table
INSERT OVERWRITE TABLE workspace.promotion_sql.silver_fact_sales
SELECT * FROM full_refresh_silver;

-- Incremental: Merge new/updated records only if mode is incremental
MERGE INTO workspace.promotion_sql.silver_fact_sales AS target
USING incremental_silver AS source
ON target.ProductKey = source.ProductKey 
   AND target.StoreKey = source.StoreKey 
   AND target.DateKey = source.DateKey
   AND target.PromotionKey = source.PromotionKey
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- Log silver stage
INSERT INTO workspace.promotion_sql.pipeline_run_log
SELECT
  DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMddHHmmss') AS run_id,
  LOWER(TRIM('${mode}')) AS run_mode,
  'silver_sales' AS stage,
  COUNT(*) AS rows_out,
  'SUCCESS' AS status,
  '' AS error_msg,
  CURRENT_TIMESTAMP() AS started_at,
  CURRENT_TIMESTAMP() AS finished_at
FROM workspace.promotion_sql.silver_fact_sales;

SELECT 
  'Sales fact created' AS status, 
  LOWER(TRIM('${mode}')) AS mode,
  COUNT(*) AS rows 
FROM silver_fact_sales;

In [0]:
%sql
-- Create Gold Price Elasticity table
CREATE OR REPLACE TABLE workspace.promotion_sql.gold_price_elasticity
USING DELTA AS
WITH weekly_agg AS (
    SELECT
        f.ProductKey, p.Product, p.UnitCost,
        f.PromotionKey, pr.PromotionName, pr.OnFlyer, pr.DiscountTier,
        f.ActualPrice, f.DiscountPct, f.Year, f.WeekNumber,
        SUM(f.UnitsSold) AS ChainUnits,
        SUM(f.SalesAmt) AS ChainSalesAmt,
        SUM(f.GrossMargin) AS ChainGrossMargin,
        SUM(f.Transactions) AS ChainTransactions,
        SUM(f.IsBelowCost) AS StoresBelowCost,
        COUNT(f.StoreKey) AS StoresActive
    FROM workspace.promotion_sql.silver_fact_sales f
    JOIN workspace.promotion_sql.silver_dim_product p ON f.ProductKey = p.ProductKey
    JOIN workspace.promotion_sql.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
    GROUP BY ALL
),
product_baseline AS (
    SELECT ProductKey,
        AVG(CASE WHEN DiscountPct = 0 THEN ActualPrice END) AS BasePrice,
        AVG(CASE WHEN DiscountPct = 0 THEN ChainUnits END) AS BaseUnits
    FROM weekly_agg
    GROUP BY ProductKey
),
elasticity_calc AS (
    SELECT w.*, b.BasePrice, b.BaseUnits,
        CASE WHEN b.BasePrice > 0 AND w.ActualPrice > 0 THEN
            ((w.ActualPrice - b.BasePrice) / b.BasePrice) * 100
        ELSE 0 END AS PriceChangePct,
        CASE WHEN b.BaseUnits > 0 AND w.ChainUnits > 0 THEN
            ((w.ChainUnits - b.BaseUnits) / b.BaseUnits) * 100
        ELSE 0 END AS UnitsChangePct
    FROM weekly_agg w
    LEFT JOIN product_baseline b ON w.ProductKey = b.ProductKey
)
SELECT *,
    CASE WHEN PriceChangePct <> 0 THEN UnitsChangePct / NULLIF(PriceChangePct, 0)
    ELSE NULL END AS PriceElasticity
FROM elasticity_calc;

SELECT 'Price elasticity created' AS status, COUNT(*) AS rows FROM gold_price_elasticity;

In [0]:
%sql
-- Create Gold Promotion Performance table
CREATE OR REPLACE TABLE workspace.promotion_sql.gold_promotion_performance
USING DELTA AS
SELECT
    pr.PromotionKey, pr.PromotionName, pr.OnFlyer,
    pr.Discount AS PromotionDiscount, pr.DiscountTier,
    p.ProductKey, p.Product, p.Brand, p.Category,
    COUNT(DISTINCT CONCAT(f.Year, '-', f.WeekNumber)) AS WeeksActive,
    COUNT(DISTINCT f.StoreKey) AS StoresUsed,
    SUM(f.UnitsSold) AS TotalUnits,
    SUM(f.SalesAmt) AS TotalSales,
    SUM(f.GrossMargin) AS TotalGrossMargin,
    AVG(f.ActualPrice) AS AvgPrice,
    SUM(f.UnitsSold) / NULLIF(COUNT(DISTINCT f.StoreKey), 0) AS UnitsPerStore,
    SUM(f.GrossMargin) / NULLIF(SUM(f.SalesAmt), 0) AS MarginPct,
    SUM(CASE WHEN f.IsBelowCost = 1 THEN 1 ELSE 0 END) AS StoresBelowCost
FROM workspace.promotion_sql.silver_fact_sales f
JOIN workspace.promotion_sql.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
JOIN workspace.promotion_sql.silver_dim_product p ON f.ProductKey = p.ProductKey
GROUP BY ALL;

SELECT 'Promotion performance created' AS status, COUNT(*) AS rows FROM gold_promotion_performance;

In [0]:
%sql
-- Create Gold Store Performance table
CREATE OR REPLACE TABLE workspace.promotion_sql.gold_store_performance
USING DELTA AS
SELECT
    s.StoreKey, s.StoreID, s.StoreName, s.City, s.Province, s.Country,
    COUNT(DISTINCT CONCAT(f.Year, '-', f.WeekNumber)) AS WeeksActive,
    COUNT(DISTINCT f.ProductKey) AS UniqueProducts,
    SUM(f.UnitsSold) AS TotalUnits,
    SUM(f.SalesAmt) AS TotalSales,
    SUM(f.GrossMargin) AS TotalGrossMargin,
    SUM(f.Transactions) AS TotalTransactions,
    SUM(f.GrossMargin) / NULLIF(SUM(f.SalesAmt), 0) AS MarginPct,
    SUM(f.SalesAmt) / NULLIF(COUNT(DISTINCT CONCAT(f.Year, '-', f.WeekNumber)), 0) AS AvgWeeklySales,
    SUM(CASE WHEN f.DiscountPct > 0 THEN f.UnitsSold ELSE 0 END) / NULLIF(SUM(f.UnitsSold), 0) AS PromoUnitsPct
FROM workspace.promotion_sql.silver_fact_sales f
JOIN workspace.promotion_sql.silver_dim_store s ON f.StoreKey = s.StoreKey
GROUP BY ALL;

SELECT 'Store performance created' AS status, COUNT(*) AS rows FROM gold_store_performance;

In [0]:
%sql
-- Display pipeline summary
SELECT '════ PIPELINE COMPLETED ════' AS summary;

SELECT 'Bronze' AS layer, 'bronze_product' AS table_name, COUNT(*) AS row_count FROM workspace.promotion_sql.bronze_product
UNION ALL
SELECT 'Bronze', 'bronze_store', COUNT(*) FROM workspace.promotion_sql.bronze_store
UNION ALL
SELECT 'Bronze', 'bronze_date', COUNT(*) FROM workspace.promotion_sql.bronze_date
UNION ALL
SELECT 'Bronze', 'bronze_promotion', COUNT(*) FROM workspace.promotion_sql.bronze_promotion
UNION ALL
SELECT 'Bronze', 'bronze_sales', COUNT(*) FROM workspace.promotion_sql.bronze_sales
UNION ALL
SELECT 'Silver', 'silver_dim_product', COUNT(*) FROM workspace.promotion_sql.silver_dim_product
UNION ALL
SELECT 'Silver', 'silver_dim_store', COUNT(*) FROM workspace.promotion_sql.silver_dim_store
UNION ALL
SELECT 'Silver', 'silver_dim_date', COUNT(*) FROM workspace.promotion_sql.silver_dim_date
UNION ALL
SELECT 'Silver', 'silver_dim_promotion', COUNT(*) FROM workspace.promotion_sql.silver_dim_promotion
UNION ALL
SELECT 'Silver', 'silver_fact_sales', COUNT(*) FROM workspace.promotion_sql.silver_fact_sales
UNION ALL
SELECT 'Gold', 'gold_price_elasticity', COUNT(*) FROM workspace.promotion_sql.gold_price_elasticity
UNION ALL
SELECT 'Gold', 'gold_promotion_performance', COUNT(*) FROM workspace.promotion_sql.gold_promotion_performance
UNION ALL
SELECT 'Gold', 'gold_store_performance', COUNT(*) FROM workspace.promotion_sql.gold_store_performance
ORDER BY layer, table_name;